# Cortex Agent — Multi-Tool Orchestration

**Cortex Agent** combines multiple tools (Analyst, Search, custom functions) into an
intelligent orchestrator that decides which tool to use based on the question.

| Item | Detail |
|---|---|
| **Capability** | Multi-tool AI agent with automatic routing |
| **Tools** | Cortex Search (retrieval), Cortex Analyst (SQL), custom functions |
| **Architecture** | Question → Agent → tool selection → execution → response |

> **Prerequisites:** Complete the **Cortex Search RAG** and **Cortex Analyst** labs first.


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Understanding Cortex Agent Architecture

Cortex Agent is accessed via the **REST API** and uses Server-Sent Events (SSE) streaming.

```
User Question
     │
     ▼
Cortex Agent (REST API)
     │
     ├── Tool 1: Cortex Search (knowledge retrieval)
     ├── Tool 2: Cortex Analyst (SQL generation)
     └── Tool 3: Custom functions
     │
     ▼
Unified Response (streamed via SSE)
```

### Key Concepts
- **Tools**: Configured resources the agent can invoke (search services, semantic models, etc.)
- **Routing**: The LLM decides which tool(s) to use based on the question
- **SSE Streaming**: Responses arrive as Server-Sent Events for real-time display


## Step 3 — Verify Available Tools

Confirm the prerequisite services exist before configuring the agent.

In [ ]:
SHOW CORTEX SEARCH SERVICES IN SCHEMA GENAI_LEARNING.PUBLIC;

In [ ]:
LIST @GENAI_LEARNING.PUBLIC.semantic_models;

## Step 4 — Agent Conversation Tracking

Create a table to log agent interactions for analysis.

In [ ]:
CREATE OR REPLACE TABLE agent_conversations (
    conversation_id VARCHAR(50) DEFAULT UUID_STRING(),
    turn_number     INT,
    user_question   TEXT,
    agent_response  TEXT,
    tools_used      VARCHAR(500),
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

## Step 5 — Calling the Cortex Agent REST API

The Cortex Agent API requires a POST request with tool configuration.
Below is the API specification — use this from any HTTP client (Postman, curl, Python, etc.).

### Endpoint
```
POST https://<account>.snowflakecomputing.com/api/v2/cortex/agent:run
```

### Request Body Structure
```json
{
  "model": "claude-3-5-sonnet",
  "messages": [
    {"role": "user", "content": [{"type": "text", "text": "Your question here"}]}
  ],
  "tools": [
    {
      "tool_spec": {
        "type": "cortex_search",
        "name": "wiki_search",
        "spec": {
          "service_name": "GENAI_LEARNING.PUBLIC.WIKI_SEARCH",
          "max_results": 3
        }
      }
    },
    {
      "tool_spec": {
        "type": "cortex_analyst",
        "name": "sales_analyst",
        "spec": {
          "semantic_model_file": "@GENAI_LEARNING.PUBLIC.semantic_models/sales_semantic_model.yaml"
        }
      }
    }
  ],
  "tool_choice": {"type": "auto"}
}
```

### Example Questions to Try
- **Search tool**: "What is a vector database?" (routes to Cortex Search)
- **Analyst tool**: "What was total revenue in 1995?" (routes to Cortex Analyst)
- **Both tools**: "How do Snowflake AI features relate to our sales performance?" (multi-tool)


## Step 6 — SQL Queries the Agent Would Generate

These are examples of SQL that the Cortex Analyst tool would generate when invoked by the agent.

In [ ]:
-- Agent receives: "What are the top shipping modes by revenue?"
-- Analyst tool generates:
SELECT
    l.ship_mode,
    SUM(l.extended_price) AS total_revenue,
    COUNT(DISTINCT l.order_key) AS order_count
FROM lineitem_vw l
GROUP BY l.ship_mode
ORDER BY total_revenue DESC;

In [ ]:
-- Agent receives: "Which market segments have the most high-value orders?"
-- Analyst tool generates:
SELECT
    c.market_segment,
    COUNT(*) AS high_value_orders,
    SUM(o.total_price) AS total_value
FROM orders_vw o
JOIN customer_vw c ON o.customer_key = c.customer_key
WHERE o.total_price > 100000
GROUP BY c.market_segment
ORDER BY high_value_orders DESC;

## Key Takeaways

- Cortex Agent orchestrates multiple AI tools via a single REST API.
- **Tool routing** is automatic — the LLM decides which tool fits the question.
- Combine Cortex Search (knowledge) + Cortex Analyst (data) for comprehensive answers.
- Use `tool_choice: {"type": "auto"}` for intelligent routing, or force a specific tool.
- SSE streaming enables real-time response rendering in applications.
- Build production apps with Streamlit + Cortex Agent for interactive AI assistants.
